# APPLYING ANN

#WITH FS

## Using Filtering Method(Constant,Quasiconstant Removal)

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/HR-Preg .csv')

# Display the first few rows of the dataset
print(df.head())

# Check for constant and quasi-constant features
constant_features = [col for col in df.columns if df[col].nunique() == 1]
quasi_constant_features = [col for col in df.columns if df[col].nunique() == 2 and (df[col].value_counts(normalize=True).max() > 0.99)]

# Combine the lists of constant and quasi-constant features
constant_and_quasi_constant_features = constant_features + quasi_constant_features

# Drop the constant and quasi-constant features
df_filtered = df.drop(columns=constant_and_quasi_constant_features)

# Display the selected important features
print("Selected important features after removing constant and quasi-constant features:")
print(df_filtered.columns)

          Name  Age Gravida TiTi Tika Gestation Period Weight Height  \
0    Rituporna   18     1st       1st          38 week  50 kg  5.3''   
1        Moina   25     2nd       2nd          38 week  60 kg  5.2''   
2       Rabeya   20     1st       1st          30 week  55 kg  5.0''   
3       Shorna   22     1st       3rd          35 week  51 kg  5.4''   
4  Tania Akter   20     1st       2nd          30 week  53 kg  5.2''   

   Systolic Pressure  Diastolic Pressure  Anemia  Jaundice Fetal Position  \
0                100                  60       0         0         Normal   
1                100                  70       0         0         Normal   
2                100                  60       0         0         Normal   
3                110                  65       0         0         Normal   
4                100                  55       0         0         Normal   

  Fetal Movements Fetal Heartbeat  Urine Test Albumin Urine Test Sugar  \
0          Normal            1

IMPLEMENTATION OF ANN

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Load the dataset
file_path = "/content/HR-Preg .csv"
df = pd.read_csv(file_path)

# Selected features after constant & quasi-constant removal
selected_features = ['Name','Age', 'Gravida', 'TiTi Tika', 'Gestation Period', 'Weight', 'Height',
                     'Systolic Pressure', 'Diastolic Pressure', 'Anemia', 'Jaundice',
                     'Fetal Heartbeat', 'Urine Test Albumin', 'Urine Test Sugar', 'VDRL', 'HRsAG', 'High-Risk Pregnancy']

df = df[selected_features]

# Convert categorical variables into numerical values
for column in df.select_dtypes(include=['object']):
    df[column] = LabelEncoder().fit_transform(df[column])

# Separate features and target
X = df.drop(columns=['High-Risk Pregnancy'])
y = df['High-Risk Pregnancy']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define ANN model
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_data=(X_test, y_test), verbose=1)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'ANN Model Accuracy: {accuracy * 100:.2f}%')


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.6200 - loss: 0.6760 - val_accuracy: 0.7100 - val_loss: 0.5877
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6554 - loss: 0.6094 - val_accuracy: 0.7400 - val_loss: 0.5455
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6787 - loss: 0.5760 - val_accuracy: 0.7700 - val_loss: 0.5139
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6861 - loss: 0.5261 - val_accuracy: 0.7550 - val_loss: 0.4837
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7360 - loss: 0.5108 - val_accuracy: 0.7750 - val_loss: 0.4579
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7356 - loss: 0.4926 - val_accuracy: 0.7850 - val_loss: 0.4357
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7896 - loss: 0.4396 - val_accuracy: 0.8000 - val_loss: 0.4142
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8029 - loss: 0.4025 - val_accuracy: 0.7800 - val_loss: 0.4015
Epo

## Using Pearson Correlation(Remove Highly Correlated Features)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from keras.models import Sequential
from keras.layers import Dense

# Load the dataset
df = pd.read_csv('/content/HR-Preg .csv')

# Separate features and target
X = df.drop('High-Risk Pregnancy', axis=1)  # Features
y = df['High-Risk Pregnancy']  # Target

# Convert target variable to numerical using Label Encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Perform one-hot encoding on the features only
X = pd.get_dummies(X, drop_first=True)

# Calculate Pearson Correlation matrix
corr_matrix = X.corr().abs()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find features with correlation greater than 0.8
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

# Drop highly correlated features
X_reduced = X.drop(to_drop, axis=1)

# Enlist the important features (remaining features after removing highly correlated ones)
important_features = X_reduced.columns.tolist()
print("Important Features after removing highly correlated features:")
print(important_features)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_reduced, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build the ANN model
model = Sequential()
model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=1)

# Evaluate the model
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy of the ANN model: {accuracy * 100:.2f}%')

Important Features after removing highly correlated features:
['Age', 'Systolic Pressure', 'Diastolic Pressure', 'Anemia', 'Jaundice', 'Urine Test Albumin', 'Name_Afrina', 'Name_Afroz', 'Name_Afroza', 'Name_Afsana', 'Name_Afshana', 'Name_Akhi Moni', 'Name_Aklima', 'Name_Aklima Akter', 'Name_Alia', 'Name_Alisa', 'Name_Alpona', 'Name_Alpona Akter', 'Name_Amina', 'Name_Anika', 'Name_Anjona', 'Name_Anju', 'Name_Anjuman', 'Name_Antora', 'Name_Anuwara', 'Name_Arifa', 'Name_Arisha', 'Name_Arjina', 'Name_Asha', 'Name_Ashma', 'Name_Asia', 'Name_Asia Jannat', 'Name_Asma', 'Name_Asmani', 'Name_Asmaul Husna', 'Name_Ayat', 'Name_Ayesha', 'Name_Azmia Sristy', 'Name_Banesa', 'Name_Benu', 'Name_Beuty', 'Name_Bilkis', 'Name_Bilkis Akter', 'Name_Bina', 'Name_Bina Akter', 'Name_Bipasha', 'Name_Bipesha', 'Name_Bithi', 'Name_Boishakhi', 'Name_Bonna', 'Name_Bristi', 'Name_Bristy', 'Name_Chadni', 'Name_Chaina Akter', 'Name_Dilara', 'Name_Easmin', 'Name_Eayasmin', 'Name_Era', 'Name_Eti', 'Name_Fahima', 'Name_

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5621 - loss: 0.6976
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7116 - loss: 0.5626
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7883 - loss: 0.4683
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8573 - loss: 0.3706
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9439 - loss: 0.2573
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9491 - loss: 0.2010
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9749 - loss: 0.1414
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9828 - loss: 0.0889
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9889 - loss: 0.0671
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9912 - loss: 0.0572
Epoch 11/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9988 - loss: 0.0388
Epoch 12/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9992 - loss: 0.0309


## Using Recursive Feature Elimination(using Random Forest)

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load the dataset
data = pd.read_csv('/content/HR-Preg .csv')

# Encode categorical variables
label_encoders = {}
for column in data.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    data[column] = le.fit_transform(data[column])
    label_encoders[column] = le

# Separate features and target variable
X = data.drop('High-Risk Pregnancy', axis=1)
y = data['High-Risk Pregnancy']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize the Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Initialize RFE with the Random Forest classifier and select the top 10 features
rfe = RFE(estimator=rf, n_features_to_select=10)

# Fit RFE
rfe.fit(X_train, y_train)

# Get the selected features
selected_features = X.columns[rfe.support_]

# Print the selected features
print("Selected Features:")
print(selected_features)

Selected Features:
Index(['Name', 'Age', 'TiTi Tika', 'Gestation Period', 'Weight', 'Height',
       'Systolic Pressure', 'Diastolic Pressure', 'VDRL', 'HRsAG'],
      dtype='object')


IMPLEMENTING ANN

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Load the dataset
data = pd.read_csv('/content/HR-Preg .csv')

# Select the important features
selected_features = ['Name', 'Age', 'TiTi Tika', 'Gestation Period', 'Weight', 'Height',
                     'Systolic Pressure', 'Diastolic Pressure', 'VDRL', 'HRsAG']
X = data[selected_features]
y = data['High-Risk Pregnancy']

# Encode categorical variables using One-Hot Encoding for 'TiTi Tika' and 'Gestation Period'
X = pd.get_dummies(X, columns=['TiTi Tika', 'Gestation Period']) # Use get_dummies for these columns

# Encode other categorical variables with Label Encoding
label_encoder = LabelEncoder()
for col in ['Name', 'VDRL', 'HRsAG','Weight','Height']:  # Include other categorical columns if needed
    X[col] = label_encoder.fit_transform(X[col])

# Encode the target variable
y = label_encoder.fit_transform(y)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build the ANN model
model = Sequential()
model.add(Dense(units=64, activation='relu', input_dim=X_train.shape[1]))
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {accuracy*100:.4f}%')

Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6662 - loss: 0.6223 - val_accuracy: 0.7250 - val_loss: 0.5808
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7527 - loss: 0.5349 - val_accuracy: 0.7250 - val_loss: 0.5391
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7596 - loss: 0.4928 - val_accuracy: 0.7188 - val_loss: 0.5192
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8100 - loss: 0.4300 - val_accuracy: 0.7437 - val_loss: 0.5097
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8136 - loss: 0.4040 - val_accuracy: 0.7500 - val_loss: 0.5005
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8136 - loss: 0.4088 - val_accuracy: 0.7625 - val_loss: 0.4894
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8074 - loss: 0.4004 - val_accuracy: 0.7625 - val_loss: 0.4816
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8090 - loss: 0.3870 - val_accuracy: 0.7688 - val_loss: 0.4